# Plant Disease Detection — Phase 1: Data Access & EDA

Capstone: **Plant Disease Detection using CNN and Transfer Learning** (PlantVillage).

**Run this on Google Colab with a GPU** TFDS will
download PlantVillage (~1 GB) on first use.

This notebook covers deliverables *Data access + preprocessing* and *EDA*:
1. Load PlantVillage and inspect the 38 classes
2. Class distribution + sample images
3. Build resized/augmented `train / val / test` tf.data pipelines


## 0. Install (Colab)

In [ ]:
# Uncomment on Colab / fresh env:
# !pip install -q tensorflow tensorflow-datasets matplotlib seaborn scikit-learn


## 1. Imports & config

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers

AUTOTUNE = tf.data.AUTOTUNE
IMG_SIZE = 224          # input size for MobileNetV2
BATCH_SIZE = 32
SEED = 42
tf.random.set_seed(SEED); np.random.seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "none (training will be slow — use Colab GPU)")


## 2. Load PlantVillage & inspect classes

In [ ]:
ds, info = tfds.load("plant_village", split="train", as_supervised=True, with_info=True)

num_examples = info.splits["train"].num_examples
class_names = info.features["label"].names
num_classes = len(class_names)

print(f"Total images : {num_examples}")
print(f"Classes      : {num_classes}")
for i, name in enumerate(class_names):
    print(f"  {i:2d}  {name}")


### Reduced-scope fallback (tomato-only)

If you need to shrink scope, filter to tomato classes here and reuse everything below.
Leave commented for the full 38-class run.


In [ ]:
# tomato_ids = [i for i, n in enumerate(class_names) if n.lower().startswith("tomato")]
# print("Tomato classes:", [class_names[i] for i in tomato_ids])
# ds = ds.filter(lambda img, lbl: tf.reduce_any(tf.equal(lbl, tomato_ids)))
# NOTE: if you do this, remap labels to 0..k-1 before training.


## 3. Class distribution

In [ ]:
counts = np.zeros(num_classes, dtype=int)
for _, labels in tfds.as_numpy(ds.batch(256)):
    counts += np.bincount(labels, minlength=num_classes)

order = np.argsort(counts)
plt.figure(figsize=(9, 10))
plt.barh([class_names[i] for i in order], counts[order])
plt.title("PlantVillage — images per class")
plt.xlabel("count"); plt.tight_layout(); plt.show()

print(f"Largest class : {counts.max()}  |  Smallest class: {counts.min()}")
print(f"Imbalance ratio (max/min): {counts.max() / max(counts.min(), 1):.1f}x")


## 4. Sample images

In [ ]:
plt.figure(figsize=(12, 8))
for i, (img, label) in enumerate(tfds.as_numpy(ds.take(15))):
    ax = plt.subplot(3, 5, i + 1)
    ax.imshow(img); ax.axis("off")
    ax.set_title(class_names[label].replace("___", "\n"), fontsize=7)
plt.tight_layout(); plt.show()


## 5. Train / val / test split + tf.data pipeline

PlantVillage ships as a single `train` split, so we carve out 80 / 10 / 10 with the TFDS
subsplit syntax. Preprocessing uses MobileNetV2's `preprocess_input`; augmentation is
applied to the training set only.

> Note: we deliberately do **not** `.cache()` the resized float tensors — at 224x224 the
> full set would exhaust RAM. `prefetch` keeps the pipeline fast enough.


In [ ]:
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

(train_raw, val_raw, test_raw) = tfds.load(
    "plant_village",
    split=["train[:80%]", "train[80%:90%]", "train[90%:]"],
    as_supervised=True,
)

def prep(img, label):
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = preprocess_input(tf.cast(img, tf.float32))
    return img, label

augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="augment")

train_ds = (train_raw.map(prep, AUTOTUNE)
            .shuffle(1000, seed=SEED)
            .batch(BATCH_SIZE)
            .map(lambda x, y: (augment(x, training=True), y), AUTOTUNE)
            .prefetch(AUTOTUNE))
val_ds  = val_raw.map(prep, AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
test_ds = test_raw.map(prep, AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

# sanity check
xb, yb = next(iter(train_ds))
print("batch images:", xb.shape, xb.dtype, "| range:", float(tf.reduce_min(xb)), "to", float(tf.reduce_max(xb)))
print("batch labels:", yb.shape)


## Next

Pipelines are ready. Move to **Phase 2 (modeling)** in `BUILD_PLAN.md`:
factor this pipeline into `src/data.py`, then build the baseline CNN and the MobileNetV2
transfer-learning model in `src/model.py` / `src/train.py`.
